In [ ]:
# CELL 1 — Connection and previously frozen experiment suite
# Paste your existing private sf_options = {...} setup above this check.
# Use the same approved compute that connected to Snowflake for RUN_001.
# No passwords, keys or tokens are included in these files.
if "sf_options" not in globals():
    raise RuntimeError("Add the existing private sf_options connection setup at the top of this cell.")
sf_options_dl_poc = sf_options.copy()
sf_options_dl_poc.update({"sfDatabase": "DSVC_TAKEDA_TA_PRIVATE", "sfSchema": "DS_ML"})
PREFIX = "TAK861_TX_READY_V63_DL_POC"
EXPERIMENT_PREFIX = PREFIX + "_BRIAN_V2"
REFERENCE_RUN_ID = "RUN_001"  # Original saved model anchors input/split fingerprints.
TARGET_SPECIFICATION = {
    "business_objective": "Rank patients by likelihood of advanced-therapy escalation in the next 90 days",
    "prediction_horizon_days": 90,
    "target_column": "RESP",
    "source": "Client slide Predicting Escalation Risk, shared from the call around 12:30",
    "label_construction_verified_against_business_definition": False,
    "status": "Business definition recorded; existing RESP is preserved without asserting its construction matches the 90-day horizon",
}

SUITE_ID = "BRIAN_001"  # New name for any changed settings, feature rules or training seed.
import re
if not re.fullmatch(r"[A-Z][A-Z0-9_]{0,19}", SUITE_ID):
    raise ValueError("SUITE_ID must use 1-20 uppercase letters, digits or underscores, starting with a letter.")
SELECTION_TABLE = f"{EXPERIMENT_PREFIX}_SELECTION_{SUITE_ID}"
PREPARATION_TABLE = EXPERIMENT_PREFIX + "_PREPARATION"
SPLIT_AUDIT_TABLE = EXPERIMENT_PREFIX + "_SPLIT_AUDIT"

print("Evaluate only the run already selected in notebook 03 using validation AP.")
print("This reuses the previously inspected TEST cohort; fresh confirmation remains a later step.")


In [ ]:
# CELL 2 — Self-contained restoration and reporting helpers
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
try:
    import torch
    import sklearn
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
except ModuleNotFoundError as error:
    raise RuntimeError("Use an approved runtime with PyTorch, scikit-learn, NumPy, pandas and matplotlib.") from error
print("PyTorch:", torch.__version__, "| scikit-learn:", sklearn.__version__, "| CUDA:", torch.cuda.is_available())
"""Embedded by the delivery notebooks; no repository dependency at runtime."""
import base64
import hashlib
import io
import json
import math
import re
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd


def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def read_sf(suffix):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", f"{PREFIX}_{suffix}").load())


def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)


def checked_metadata(frame, include_split=False):
    columns = ["PATIENT_ID", "END_DT", "RESP"]
    if include_split:
        columns += ["SPLIT", "SPLIT_CONFIG"]
    out = frame.loc[:, columns].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Snapshot metadata must be nonempty and contain no nulls.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x)).all():
        raise ValueError("PATIENT_ID must retain its original nonempty string value.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("END_DT must be a date without an intraday time/timezone.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("RESP must be exactly 0 or 1 before conversion.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys in snapshot metadata.")
    if include_split:
        if set(out.SPLIT) != {"train", "validation", "test"}:
            raise ValueError("Expected the saved train, validation and test assignments.")
        if out.groupby("PATIENT_ID", observed=True).SPLIT.nunique().gt(1).any():
            raise ValueError("Patient overlap between splits.")
        if len(out.SPLIT_CONFIG.unique()) != 1:
            raise ValueError("The split contains inconsistent creation settings.")
        for _, part in out.groupby("SPLIT", observed=True):
            if set(part.RESP) != {0, 1}:
                raise ValueError("Each split must contain both response classes.")
    return out.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)


def validate_metadata_pair(snapshot_frame, manifest_frame, features):
    source = checked_metadata(snapshot_frame)
    manifest = checked_metadata(manifest_frame, include_split=True)
    if not source.equals(manifest[["PATIENT_ID", "END_DT", "RESP"]]):
        raise ValueError("Frozen split and source snapshots differ in keys or labels.")
    creation = json.loads(manifest.SPLIT_CONFIG.iloc[0])
    # Match the feature-order hash created in the completed split notebook.
    feature_order_hash = hashlib.sha256(
        json.dumps(features, ensure_ascii=False).encode("utf-8")).hexdigest()
    if creation.get("feature_order_sha256") != feature_order_hash or creation.get("n_timesteps") != 12:
        raise ValueError("Feature order or timesteps differ from the frozen split.")
    return manifest, creation


def new_tensor_buffer(metadata, feature_count, seq_len=12):
    temporary = tempfile.TemporaryDirectory(prefix="dl_poc_")
    path = Path(temporary.name) / "counts.float32"
    X = np.memmap(path, dtype="<f4", mode="w+", shape=(len(metadata), seq_len, feature_count))
    return temporary, X


def fill_tensor(X, metadata, sequence_rows):
    """Place rows by canonical keys, independent of Spark partition order."""
    positions = {(r.PATIENT_ID, r.END_DT): i for i, r in enumerate(metadata.itertuples())}
    seen = np.zeros(len(metadata), dtype=bool)
    seq_len, feature_count = X.shape[1:]
    for row in sequence_rows:
        raw_date = row["END_DT"]
        if raw_date is None or row["PATIENT_ID"] is None:
            raise ValueError("Null monthly snapshot key.")
        date = pd.Timestamp(raw_date)
        if date.tzinfo is not None or date != date.normalize():
            raise ValueError("Monthly END_DT is not an exact date.")
        key = (row["PATIENT_ID"], date.strftime("%Y-%m-%d"))
        if key not in positions:
            raise ValueError("Unexpected monthly snapshot key.")
        i = positions[key]
        if seen[i] or row["RESP"] != int(metadata.RESP.iloc[i]):
            raise ValueError("Duplicate monthly snapshot or changed label.")
        sequence = row["SEQUENCE"]
        steps = [month["T"] for month in sequence]
        if len(sequence) != seq_len or any(t is None for t in steps):
            raise ValueError("Expected exactly 12 complete timesteps per snapshot.")
        # Check original values before any integer conversion.
        if sorted(steps) != list(range(seq_len)):
            raise ValueError("Timesteps must be unique integers 0 through 11.")
        sequence = sorted(sequence, key=lambda month: month["T"])
        values = np.asarray([month["V"] for month in sequence], dtype=np.float32)
        if values.shape != (seq_len, feature_count):
            raise ValueError("Monthly feature shape does not match the vocabulary.")
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError("Counts must be finite, nonnegative float32 values with no nulls.")
        X[i] = values
        seen[i] = True
    if not seen.all():
        raise ValueError("Monthly data is missing original snapshots, including zero-activity sequences.")
    X.flush()


def input_fingerprints(X, metadata, features):
    tensor_hash = hashlib.sha256()
    tensor_hash.update(canonical_json(list(X.shape)).encode("utf-8"))
    for start in range(0, len(X), 128):
        tensor_hash.update(np.asarray(X[start:start + 128], dtype="<f4").tobytes(order="C"))
    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP), str(r.SPLIT)]
               for r in metadata.itertuples()]
    return {"model_input_sha256": tensor_hash.hexdigest(),
            "snapshot_manifest_sha256": digest_json(records),
            "feature_names_sha256": digest_json(features),
            "split_config_sha256": digest_json(json.loads(metadata.SPLIT_CONFIG.iloc[0]))}


def load_inputs():
    from pyspark.sql import functions as F
    mapping = read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()
    if not mapping or [r["FEATURE_INDEX"] for r in mapping] != list(range(len(mapping))):
        raise ValueError("Invalid feature indices.")
    features = [r["FEATURE_NAME"] for r in mapping]
    aliases = [r["FEATURE_COLUMN"] for r in mapping]
    if (aliases != [f"F{i:04d}" for i in range(len(features))]
            or not all(isinstance(f, str) and f for f in features)
            or len(set(features)) != len(features)):
        raise ValueError("Invalid feature names, aliases or order.")
    source = read_sf("SNAPSHOTS").select("PATIENT_ID", "END_DT", "RESP").toPandas()
    frozen = read_sf("PATIENT_SPLIT").select(
        "PATIENT_ID", "END_DT", "RESP", "SPLIT", "SPLIT_CONFIG").toPandas()
    metadata, creation = validate_metadata_pair(source, frozen, features)
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()), len(features))
    if observed != (23151, 12447, 1345, 1028):
        raise ValueError(f"V63 population changed: snapshots/patients/positives/features = {observed}.")
    monthly = read_sf("TENSOR_MONTHLY")
    if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):
        raise ValueError("Monthly table columns differ from the frozen feature map.")
    print("Building the model input in a temporary driver file (about 1.06 GiB).", flush=True)
    grouped = (monthly.groupBy("PATIENT_ID", "END_DT", "RESP")
        .agg(F.collect_list(F.struct(
            F.col("TIME_STEP").alias("T"),
            F.array(*[F.when(F.col(name) >= 0, F.col(name).cast("float"))
                      .otherwise(F.lit(None).cast("float")) for name in aliases]).alias("V")
        )).alias("SEQUENCE"))
        .repartition(128))
    temporary, X = new_tensor_buffer(metadata, len(features))
    try:
        fill_tensor(X, metadata, grouped.toLocalIterator(prefetchPartitions=False))
        hashes = input_fingerprints(X, metadata, features)
        X.flags.writeable = False
    except BaseException:
        X._mmap.close()
        temporary.cleanup()
        raise
    y = metadata.RESP.to_numpy(dtype=np.float32)
    indices = {name: np.flatnonzero(metadata.SPLIT.to_numpy() == name)
               for name in ("train", "validation", "test")}
    print(f"Validated tensor shape {X.shape}; patient overlap = 0.", flush=True)
    return {"X": X, "y": y, "metadata": metadata, "features": features,
            "indices": indices, "hashes": hashes, "split_creation": creation,
            "temporary_directory": temporary}


def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows


def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result


ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]


def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)


def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")


"""Small sequence classifier; zero-activity months remain real timesteps."""

from dataclasses import dataclass

import torch
from torch import nn


@dataclass(frozen=True)
class ModelConfig:
    input_dim: int
    seq_len: int = 12
    d_model: int = 128
    n_heads: int = 4
    encoder_layers: int = 2
    feedforward_dim: int = 256
    dropout: float = 0.2

    def __post_init__(self):
        sizes = (self.input_dim, self.seq_len, self.d_model, self.n_heads,
                 self.encoder_layers, self.feedforward_dim)
        if any(not isinstance(n, int) or isinstance(n, bool) or n <= 0 for n in sizes):
            raise ValueError("All model dimensions must be positive integers.")
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads.")
        if not 0 <= self.dropout < 1:
            raise ValueError("dropout must be in [0, 1).")


class ClaimsTransformer(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.projection = nn.Linear(config.input_dim, config.d_model)
        self.position = nn.Parameter(torch.empty(1, config.seq_len, config.d_model))
        nn.init.normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.n_heads,
            dim_feedforward=config.feedforward_dim, dropout=config.dropout,
            activation="relu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=config.encoder_layers, enable_nested_tensor=False,
        )
        # TransformerEncoder clones the initial layer; initialize matrix weights
        # independently so the layers do not begin with identical weights.
        for encoder_layer in self.encoder.layers:
            for parameter in encoder_layer.parameters():
                if parameter.dim() > 1:
                    nn.init.xavier_uniform_(parameter)
        self.norm = nn.LayerNorm(config.d_model)
        self.head = nn.Sequential(
            nn.Linear(config.d_model, 64), nn.ReLU(),
            nn.Dropout(config.dropout), nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        expected = (self.config.seq_len, self.config.input_dim)
        if x.ndim != 3 or tuple(x.shape[1:]) != expected:
            raise ValueError(f"Expected [batch, {expected[0]}, {expected[1]}] input.")
        if not x.is_floating_point():
            raise TypeError("Transformer input must be floating point.")
        # A zero month is observed absence of activity, not padding. A causal
        # mask is unnecessary because every included month precedes the cutoff.
        hidden = self.projection(x) + self.position
        pooled = self.norm(self.encoder(hidden)).mean(dim=1)
        return self.head(pooled).squeeze(-1)


"""Classification metrics and a threshold selected only on VALIDATION."""

import numpy as np
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)


def _validate(y, probabilities):
    labels = np.asarray(y)
    raw_scores = np.asarray(probabilities)
    if np.iscomplexobj(raw_scores) or (
        raw_scores.dtype == object
        and any(isinstance(value, (complex, np.complexfloating)) for value in raw_scores.flat)
    ):
        raise ValueError("Probabilities must be real values, not complex numbers.")
    scores = np.asarray(raw_scores, dtype=float)
    if labels.ndim != 1 or scores.ndim != 1 or len(labels) != len(scores) or not len(labels):
        raise ValueError("Labels and probabilities must be aligned, nonempty 1-D arrays.")
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Labels must be binary 0/1.")
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Probabilities must be finite and in [0, 1].")
    return labels.astype(np.int64), scores


def select_validation_threshold(y, probabilities) -> float:
    """Maximize VALIDATION F1; an exact tie uses the highest threshold.

    The caller must provide VALIDATION labels and scores, never TEST. Predictions
    are positive when score >= threshold. Equal scores are never split, and
    integer cross-products identify exact F1 ties without rounding ambiguity.
    """
    labels, scores = _validate(y, probabilities)
    if len(np.unique(labels)) != 2:
        raise ValueError("Threshold selection requires both VALIDATION classes.")
    order = np.argsort(scores, kind="stable")[::-1]
    ranked_scores = scores[order]
    true_positives = np.cumsum(labels[order], dtype=np.int64)
    group_ends = np.r_[np.flatnonzero(ranked_scores[:-1] != ranked_scores[1:]), len(labels) - 1]
    total_positives = int(labels.sum())
    best_numerator, best_denominator = 0, 1
    best_threshold = float(ranked_scores[0])
    for end in group_ends:
        # F1 = 2 TP / (number selected + total positives). Python integers
        # keep cross-products exact and avoid fixed-width integer overflow.
        numerator = 2 * int(true_positives[end])
        denominator = int(end) + 1 + total_positives
        if numerator * best_denominator > best_numerator * denominator:
            best_numerator, best_denominator = numerator, denominator
            best_threshold = float(ranked_scores[end])
        # Descending thresholds retain the highest cutoff on an exact tie.
    return best_threshold


def classification_metrics(y, probabilities, threshold: float) -> dict:
    labels, scores = _validate(y, probabilities)
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("The fixed classification threshold must be in [0, 1].")
    predicted = (scores >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(labels, scores)) if labels.sum() else None,
        "roc_auc": float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


"""Seed configuration and aggregate-only runtime provenance."""

import os
import platform
import random

import numpy as np
import sklearn
import torch


def seed_everything(seed: int) -> None:
    if not isinstance(seed, int) or isinstance(seed, bool) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32).")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int) -> None:
    """Use the DataLoader's seeded generator for each worker process."""
    del worker_id
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def resolve_device(device: str = "auto") -> torch.device:
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    resolved = torch.device(device)
    if resolved.type not in {"cpu", "cuda"}:
        raise ValueError("Supported devices are auto, cpu, or cuda[:index].")
    if resolved.type == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA requested but unavailable; set device='cpu'.")
    return resolved


def runtime_metadata() -> dict:
    return {
        "python_version": platform.python_version(),
        "numpy_version": str(np.__version__),
        "sklearn_version": str(sklearn.__version__),
        "torch_version": str(torch.__version__),
        "cuda_version": str(torch.version.cuda),
        "deterministic_algorithms_requested": torch.are_deterministic_algorithms_enabled(),
        "determinism_scope": "Best effort within a fixed device and software environment; unsupported operations warn.",
    }


"""Embedded selection helpers; no repository dependency in delivered notebooks.

The supervised selector uses TRAIN only. Its importance is a screening heuristic,
not Transformer attribution: four three-month summaries preserve broad recency
but ignore order inside each window, can divide importance among correlated
features, and can favor features with more distinct count values.
Snapshots remain the fitting unit; repeated snapshots from one patient are not
independent observations. The prevalence filter counts distinct TRAIN patients.
The float32 screening matrix adds up to about 267 MB for 16,256 TRAIN snapshots
and all 1,028 source features; sklearn adds working memory. Bounded, two-pass
TRAIN reads avoid copying the full tensor or keeping two screening matrices.
"""
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier


def _fs_digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False).encode("utf-8")).hexdigest()


def _fs_positive_int(value, name):
    if type(value) is not int or value < 1:
        raise ValueError(f"{name} must be a positive Python integer.")
    return value


def _fs_seed(value):
    if type(value) is not int or not 0 <= value < 2**32:
        raise ValueError("seed must be a Python integer from 0 through 2**32 - 1.")
    return value


def _fs_source_features(data):
    features = data["features"]
    if (not isinstance(features, list) or not features
            or not all(isinstance(name, str) and name for name in features)
            or len(set(features)) != len(features)):
        raise ValueError("Source feature names must be unique, nonempty strings in a list.")
    shape = data["X"].shape
    if len(shape) != 3 or shape[1] != 12 or shape[2] != len(features):
        raise ValueError("Expected source tensor [snapshots, 12, source features].")
    source_hash = _fs_digest(features)
    recorded_hash = data.get("hashes", {}).get("feature_names_sha256")
    if recorded_hash is not None and recorded_hash != source_hash:
        raise ValueError("Source feature names differ from the validated input fingerprint.")
    return features, source_hash


def _fs_training_rows(data):
    indices = np.asarray(data["indices"]["train"])
    if (indices.ndim != 1 or not len(indices) or indices.dtype.kind not in "iu"
            or len(np.unique(indices)) != len(indices)
            or indices.min() < 0 or indices.max() >= data["X"].shape[0]):
        raise ValueError("TRAIN indices must be unique, valid integer row positions.")
    metadata = data["metadata"].iloc[indices]
    if not metadata["SPLIT"].eq("train").all():
        raise ValueError("A requested selector fitting row is not in TRAIN.")
    patients = metadata["PATIENT_ID"]
    if patients.isna().any() or not patients.map(lambda x: isinstance(x, str) and bool(x)).all():
        raise ValueError("TRAIN patient identifiers must be nonempty strings.")
    patient_codes, patient_names = pd.factorize(patients, sort=True)
    return indices, patient_codes, len(patient_names)


def _fs_ranker_settings(seed):
    return {"n_estimators": 128, "max_depth": 8, "min_samples_leaf": 10,
            "max_features": "sqrt", "class_weight": "balanced",
            "n_jobs": 2, "random_state": seed}


def _fs_temporal_settings():
    return {"aggregation": "log1p_of_four_consecutive_3_month_sums",
        "temporal_windows": [[0, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]],
        "time_step_direction": "0_newest_11_oldest",
        "flatten_order": "window_then_source_feature",
        "importance_aggregation": "sum_four_window_importances_per_feature",
        "prevalence_unit": "unique_train_patients",
        "constant_filter": "constant_in_all_four_3_month_sums"}


def _fs_train_block(data, rows, feature_count):
    block = np.asarray(data["X"][rows], dtype=np.float32)
    if block.shape != (len(rows), 12, feature_count):
        raise ValueError("A TRAIN tensor block has an unexpected shape.")
    if not np.isfinite(block).all() or (block < 0).any():
        raise ValueError("TRAIN counts must be finite and nonnegative.")
    return block


def _fs_audit(features):
    return pd.DataFrame({"source_index": range(len(features)), "feature_name": features,
        "selected": False, "selection_status": "not_selected",
        "train_nonzero_patients": pd.array([pd.NA] * len(features), dtype="Int64"),
        "train_nonzero_patient_fraction": np.nan,
        "train_aggregate_min": np.nan, "train_aggregate_max": np.nan,
        "train_variable_windows": pd.array([pd.NA] * len(features), dtype="Int64"),
        "ranking_importance": np.nan,
        "selection_rank": pd.array([pd.NA] * len(features), dtype="Int64")})


def select_features(data, mode="reduced", max_features=150, core_features=None,
                    min_train_patients=10, seed=42):
    """Return (frozen manifest, aggregate-only audit), without reading held-out values.

    ``all`` and ``core`` are fixed selections and never inspect X or y values.
    ``core`` requires exact existing tensor feature names; legacy demographic or
    engineered features need a separate upstream, date-safe feature preparation.
    Reduced selection keeps at most max_features, with no artificial padding when
    fewer survive. All modes preserve the original source order for model inputs.
    """
    features, source_hash = _fs_source_features(data)
    if mode not in {"all", "core", "reduced"}:
        raise ValueError("Feature selection mode must be all, core or reduced.")
    audit = _fs_audit(features)
    if mode == "all":
        selected = list(range(len(features)))
        settings = {"selection_method": "all_source_features"}
        fit_split = "fixed"
    elif mode == "core":
        if (not isinstance(core_features, (list, tuple)) or not core_features
                or not all(isinstance(name, str) and name for name in core_features)):
            raise ValueError("Core experiment is pending: configure a nonempty exact feature-name list.")
        if len(set(core_features)) != len(core_features):
            raise ValueError("Core feature configuration contains duplicate names.")
        missing = sorted(set(core_features) - set(features))
        if missing:
            raise ValueError(f"Core features are absent from the tensor; no name approximation is allowed: {missing}")
        requested = set(core_features)
        selected = [i for i, name in enumerate(features) if name in requested]
        settings = {"selection_method": "configured_exact_feature_names",
                    "core_features": [features[i] for i in selected]}
        fit_split = "fixed"
    else:
        _fs_positive_int(max_features, "max_features")
        _fs_positive_int(min_train_patients, "min_train_patients")
        _fs_seed(seed)
        train, patient_codes, n_patients = _fs_training_rows(data)
        # Only these explicit TRAIN row positions may be read from X or y.
        labels = np.asarray(data["y"][train])
        if labels.shape != (len(train),) or not np.isin(labels, [0, 1]).all() or len(np.unique(labels)) != 2:
            raise ValueError("Supervised selection requires binary TRAIN labels with both classes.")
        patient_present = np.zeros((n_patients, len(features)), dtype=bool)
        aggregate_min = np.full(len(features), np.inf, dtype=np.float64)
        aggregate_max = np.full(len(features), -np.inf, dtype=np.float64)
        window_min = np.full((4, len(features)), np.inf, dtype=np.float64)
        window_max = np.full((4, len(features)), -np.inf, dtype=np.float64)
        for start in range(0, len(train), 128):
            stop = min(start + 128, len(train))
            block = _fs_train_block(data, train[start:stop], len(features))
            windows = block.reshape(stop - start, 4, 3, len(features)).sum(axis=2, dtype=np.float64)
            window_min = np.minimum(window_min, windows.min(axis=0))
            window_max = np.maximum(window_max, windows.max(axis=0))
            sums = windows.sum(axis=1)
            aggregate_min = np.minimum(aggregate_min, sums.min(axis=0))
            aggregate_max = np.maximum(aggregate_max, sums.max(axis=0))
            np.logical_or.at(patient_present, patient_codes[start:stop], np.any(block > 0, axis=1))
        patient_counts = patient_present.sum(axis=0, dtype=np.int64)
        del patient_present
        variable_windows = (window_max > window_min).sum(axis=0)
        nonconstant = variable_windows > 0
        eligible = np.flatnonzero((patient_counts >= min_train_patients) & nonconstant)
        audit["train_nonzero_patients"] = pd.array(patient_counts, dtype="Int64")
        audit["train_nonzero_patient_fraction"] = patient_counts / n_patients
        audit["train_aggregate_min"] = aggregate_min
        audit["train_aggregate_max"] = aggregate_max
        audit["train_variable_windows"] = pd.array(variable_windows, dtype="Int64")
        for window in range(4):
            audit[f"train_window_{window}_sum_min"] = window_min[window]
            audit[f"train_window_{window}_sum_max"] = window_max[window]
        audit["selection_status"] = "below_importance_cutoff"
        audit.loc[patient_counts < min_train_patients, "selection_status"] = "rare_train_patients"
        audit.loc[~nonconstant, "selection_status"] = "constant_all_train_windows"
        audit.loc[patient_counts == 0, "selection_status"] = "empty_train"
        if not len(eligible):
            raise ValueError("No features survive TRAIN prevalence/constant filters; review the TRAIN feature audit/settings.")
        # Fit matrix contains eligible features only. Windows stay in actual
        # TIME_STEP order: 0-2 newest, then 3-5, 6-8, and 9-11 oldest.
        aggregates = np.empty((len(train), 4 * len(eligible)), dtype=np.float32)
        for start in range(0, len(train), 128):
            stop = min(start + 128, len(train))
            block = _fs_train_block(data, train[start:stop], len(features))
            windows = block.reshape(stop - start, 4, 3, len(features)).sum(axis=2, dtype=np.float64)
            aggregates[start:stop] = np.log1p(windows[:, :, eligible]).reshape(stop - start, -1)
        ranker_settings = _fs_ranker_settings(seed)
        ranker = ExtraTreesClassifier(**ranker_settings)
        ranker.fit(aggregates, labels.astype(np.int64))
        window_importance = np.asarray(ranker.feature_importances_, dtype=np.float64)
        if (window_importance.shape != (4 * len(eligible),)
                or not np.isfinite(window_importance).all() or (window_importance < 0).any()):
            raise ValueError("Feature ranker returned invalid importance values.")
        importance = window_importance.reshape(4, len(eligible)).sum(axis=0)
        ranking = np.lexsort((eligible, -importance))
        ranked_indices = eligible[ranking]
        audit.loc[eligible, "ranking_importance"] = importance
        audit.loc[ranked_indices, "selection_rank"] = np.arange(1, len(eligible) + 1)
        selected = sorted(int(i) for i in ranked_indices[:max_features])
        settings = {"selection_method": "train_extra_trees_impurity_importance",
            "max_features": max_features, "min_train_patients": min_train_patients, "seed": seed,
            **_fs_temporal_settings(), "ranker_params": ranker_settings,
            "n_train_snapshots": int(len(train)), "n_train_patients": int(n_patients)}
        fit_split = "train"
    audit.loc[selected, "selected"] = True
    audit.loc[selected, "selection_status"] = "selected"
    manifest = {"format_version": 1, "mode": mode,
        "source_feature_names_sha256": source_hash, "selected_indices": selected,
        "selected_features": [features[i] for i in selected],
        "settings": settings, "selection_fit_split": fit_split}
    manifest["manifest_sha256"] = _fs_digest(manifest)
    validate_feature_manifest(manifest, data)
    return manifest, audit


def validate_feature_manifest(manifest, data):
    """Validate a saved selection against source schema without refitting or reading X/y values."""
    expected = {"format_version", "mode", "source_feature_names_sha256", "selected_indices",
                "selected_features", "settings", "selection_fit_split", "manifest_sha256"}
    if not isinstance(manifest, dict) or set(manifest) != expected:
        raise ValueError("Feature manifest fields are missing or unexpected.")
    if type(manifest["format_version"]) is not int or manifest["format_version"] != 1:
        raise ValueError("Unsupported feature manifest version.")
    digest_body = {key: value for key, value in manifest.items() if key != "manifest_sha256"}
    if manifest["manifest_sha256"] != _fs_digest(digest_body):
        raise ValueError("Feature manifest SHA256 verification failed.")
    features, source_hash = _fs_source_features(data)
    if manifest["source_feature_names_sha256"] != source_hash:
        raise ValueError("Feature manifest refers to a different source vocabulary/order.")
    selected = manifest["selected_indices"]
    if (not isinstance(selected, list) or not selected
            or not all(type(i) is int and 0 <= i < len(features) for i in selected)
            or selected != sorted(set(selected))):
        raise ValueError("Selected feature indices must be unique Python integers in source order.")
    if manifest["selected_features"] != [features[i] for i in selected]:
        raise ValueError("Selected feature names do not exactly match their source indices.")
    mode, settings = manifest["mode"], manifest["settings"]
    if mode not in {"all", "core", "reduced"} or not isinstance(settings, dict):
        raise ValueError("Invalid feature manifest mode/settings.")
    if mode == "all":
        if selected != list(range(len(features))) or settings != {"selection_method": "all_source_features"}:
            raise ValueError("An all-features manifest must contain every source feature.")
    elif mode == "core":
        expected_settings = {"selection_method": "configured_exact_feature_names",
                             "core_features": [features[i] for i in selected]}
        if settings != expected_settings:
            raise ValueError("Core-feature settings do not match the exact saved selection.")
    else:
        setting_keys = {"selection_method", "max_features", "min_train_patients", "seed",
            "ranker_params", "n_train_snapshots", "n_train_patients"} | set(_fs_temporal_settings())
        if set(settings) != setting_keys:
            raise ValueError("Reduced feature selection settings are incomplete or unexpected.")
        _fs_positive_int(settings["max_features"], "max_features")
        _fs_positive_int(settings["min_train_patients"], "min_train_patients")
        _fs_seed(settings["seed"])
        _fs_positive_int(settings["n_train_snapshots"], "n_train_snapshots")
        _fs_positive_int(settings["n_train_patients"], "n_train_patients")
        fixed_settings = {"selection_method": "train_extra_trees_impurity_importance",
            **_fs_temporal_settings(),
            "ranker_params": _fs_ranker_settings(settings["seed"])}
        if any(settings[key] != value for key, value in fixed_settings.items()):
            raise ValueError("Reduced feature selection method/settings differ from this implementation.")
        train, _, n_patients = _fs_training_rows(data)
        if (settings["n_train_snapshots"] != len(train) or settings["n_train_patients"] != n_patients
                or settings["min_train_patients"] > n_patients or len(selected) > settings["max_features"]):
            raise ValueError("Reduced feature selection counts/settings do not match the source TRAIN split.")
    expected_split = "train" if mode == "reduced" else "fixed"
    if manifest["selection_fit_split"] != expected_split:
        raise ValueError("Feature selection fitting split is inconsistent with its mode.")
    return manifest


"""Selected-feature experiments, embedded after the input/model/metric helpers.

Training reads TRAIN and VALIDATION rows only. TEST scoring is a separate call.
The sequence model consumes log1p monthly counts; the logistic comparator uses
TRAIN-standardized log1p total counts and intentionally discards month order.
"""
import copy
import hashlib
import io
import json
import warnings
from dataclasses import asdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


class ExperimentSequenceDataset(Dataset):
    def __init__(self, data, split, selected_indices):
        if split not in ("train", "validation", "test"):
            raise ValueError("Unknown dataset split.")
        self.data = data
        self.rows = np.asarray(data["indices"][split], dtype=np.int64)
        self.columns = np.asarray(selected_indices, dtype=np.int64)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = int(self.rows[index])
        # Index the row first: mixed NumPy advanced indexing would swap axes.
        values = np.array(self.data["X"][row][:, self.columns], dtype=np.float32, copy=True)
        return torch.from_numpy(values), torch.tensor(float(self.data["y"][row]), dtype=torch.float32)


def _experiment_loader(data, split, columns, batch_size, seed, shuffle=False):
    return DataLoader(ExperimentSequenceDataset(data, split, columns),
                      batch_size=batch_size, shuffle=shuffle, drop_last=False,
                      num_workers=0, generator=torch.Generator().manual_seed(seed))


def _experiment_log1p(values):
    if not torch.isfinite(values).all() or (values < 0).any():
        raise ValueError("Expected finite nonnegative raw counts before log1p.")
    return torch.log1p(values)


class NumericLogisticModel(nn.Module):
    """Safe inference from numeric parameters; contains no pickled estimator."""
    def __init__(self, coefficient, intercept, scaler_mean, scaler_scale, seq_len):
        super().__init__()
        tensors = [torch.as_tensor(value, dtype=torch.float64).clone()
                   for value in (coefficient, intercept, scaler_mean, scaler_scale)]
        coefficient, intercept, scaler_mean, scaler_scale = tensors
        if (coefficient.ndim != 1 or not coefficient.numel()
                or scaler_mean.shape != coefficient.shape or scaler_scale.shape != coefficient.shape
                or intercept.shape != (1,) or (scaler_scale <= 0).any()
                or not all(torch.isfinite(value).all() for value in tensors)):
            raise ValueError("Invalid numeric logistic model/scaler parameters.")
        self.seq_len, self.input_dim = int(seq_len), len(coefficient)
        self.register_buffer("coefficient", coefficient)
        self.register_buffer("intercept", intercept)
        self.register_buffer("scaler_mean", scaler_mean)
        self.register_buffer("scaler_scale", scaler_scale)

    def forward(self, raw_monthly):
        if raw_monthly.ndim != 3 or tuple(raw_monthly.shape[1:]) != (self.seq_len, self.input_dim):
            raise ValueError("Logistic input must retain the selected monthly tensor layout.")
        if not torch.isfinite(raw_monthly).all() or (raw_monthly < 0).any():
            raise ValueError("Expected finite nonnegative raw monthly counts.")
        totals = raw_monthly.to(dtype=torch.float64).sum(dim=1)
        standardized = (_experiment_log1p(totals) - self.scaler_mean) / self.scaler_scale
        return standardized @ self.coefficient + self.intercept[0]


def _experiment_forward(model, raw, kind):
    if kind == "transformer":
        return model(_experiment_log1p(raw))
    if kind == "logistic":
        return model(raw)
    raise ValueError("Unknown model kind.")


def _experiment_predict_loader(model, loader, device, kind, criterion=None):
    model.eval()
    labels, scores, loss_sum, count = [], [], 0.0, 0
    with torch.inference_mode():
        for raw, y in loader:
            raw, y = raw.to(device), y.to(device)
            logits = _experiment_forward(model, raw, kind)
            if not torch.isfinite(logits).all():
                raise ValueError("Nonfinite model predictions.")
            if criterion is not None:
                loss = criterion(logits, y.to(logits.dtype))
                if not torch.isfinite(loss):
                    raise ValueError("Nonfinite diagnostic loss.")
                loss_sum += float(loss.item()) * len(y)
            count += len(y)
            labels.append(y.cpu().numpy())
            scores.append(torch.sigmoid(logits).cpu().numpy())
    if not count:
        raise ValueError("Cannot score an empty split.")
    return (None if criterion is None else loss_sum / count,
            np.concatenate(labels), np.concatenate(scores))


def _experiment_topk(data, split, labels, scores):
    """Same label-independent score tie ordering as the TEST report helper."""
    rows = data["indices"][split]
    frame = data["metadata"].iloc[rows][["PATIENT_ID", "END_DT"]].copy().reset_index(drop=True)
    if len(frame) != len(labels) or len(frame) < 10:
        raise ValueError("At least ten aligned snapshots are required for top-k reporting.")
    frame["END_DT"] = pd.to_datetime(frame.END_DT).dt.strftime("%Y-%m-%d")
    frame["_score"] = np.asarray(scores)
    frame["_position"] = np.arange(len(frame))
    frame["_tie"] = [hashlib.sha256(json.dumps(
        ["tak861-targeting-v1", str(patient), str(date)], separators=(",", ":")
    ).encode("utf-8")).hexdigest() for patient, date in frame[["PATIENT_ID", "END_DT"]].itertuples(index=False, name=None)]
    order = frame.sort_values(["_score", "_tie", "PATIENT_ID", "END_DT"],
                              ascending=[False, True, True, True])._position.to_numpy()
    ranked_y = np.asarray(labels)[order]
    positives, n = int(ranked_y.sum()), len(ranked_y)
    if not 0 < positives < n:
        raise ValueError("Top-k reporting requires both classes.")
    reports = []
    for percentage in (10, 20, 30):
        selected = (percentage * n + 99) // 100
        captured = int(ranked_y[:selected].sum())
        precision = captured / selected
        reports.append({"top_k_pct": percentage, "n_selected": selected,
                        "n_resp1": captured, "precision": precision,
                        "recall": captured / positives, "lift": precision / (positives / n),
                        "actual_population_fraction": selected / n})
    return reports


def _experiment_settings(settings, kind):
    settings = dict(settings)
    for name in ("batch_size",):
        if not isinstance(settings.get(name), int) or isinstance(settings[name], bool) or settings[name] <= 0:
            raise ValueError(f"{name} must be a positive integer.")
    # seed_everything performs complete seed validation before any fitting.
    if kind == "transformer":
        for name in ("epochs", "patience"):
            if not isinstance(settings.get(name), int) or isinstance(settings[name], bool) or settings[name] <= 0:
                raise ValueError(f"{name} must be a positive integer.")
        for name in ("learning_rate", "grad_clip"):
            if not np.isfinite(settings[name]) or settings[name] <= 0:
                raise ValueError(f"{name} must be positive and finite.")
        for name in ("weight_decay", "min_delta"):
            if not np.isfinite(settings[name]) or settings[name] < 0:
                raise ValueError(f"{name} must be nonnegative and finite.")
    elif kind == "logistic":
        settings.setdefault("logistic_C", 1.0)
        settings.setdefault("logistic_max_iter", 2000)
        if not np.isfinite(settings["logistic_C"]) or settings["logistic_C"] <= 0:
            raise ValueError("logistic_C must be positive and finite.")
        if (not isinstance(settings["logistic_max_iter"], int)
                or isinstance(settings["logistic_max_iter"], bool) or settings["logistic_max_iter"] <= 0):
            raise ValueError("logistic_max_iter must be a positive integer.")
    else:
        raise ValueError("model_kind must be transformer or logistic.")
    return settings


def _experiment_class_counts(data):
    counts = {}
    for split in ("train", "validation"):
        labels = np.asarray(data["y"][data["indices"][split]])
        if labels.ndim != 1 or not np.isin(labels, [0, 1]).all():
            raise ValueError("Training/validation labels must be binary.")
        positives = int(labels.sum())
        if not 0 < positives < len(labels):
            raise ValueError(f"{split} requires both classes.")
        if len(labels) < 10:
            raise ValueError(f"{split} requires at least ten snapshots for top-k metrics.")
        counts[split] = {"snapshots": len(labels), "positives": positives,
                         "negatives": len(labels) - positives}
    return counts


def _logistic_train_matrix(data, columns, batch_size):
    """Build only TRAIN aggregate rows, never a copy of the full monthly tensor."""
    rows = data["indices"]["train"]
    out = np.empty((len(rows), len(columns)), dtype=np.float64)
    loader = _experiment_loader(data, "train", columns, batch_size, 0)
    offset = 0
    for raw, _ in loader:
        if not torch.isfinite(raw).all() or (raw < 0).any():
            raise ValueError("Logistic training requires finite nonnegative counts.")
        values = np.log1p(raw.numpy().sum(axis=1, dtype=np.float64))
        out[offset:offset + len(values)] = values
        offset += len(values)
    return out


def run_experiment(data, manifest, model_config, settings, run_id, model_kind="transformer"):
    """Train one fixed feature/model experiment; select checkpoint/threshold on VAL."""
    manifest = copy.deepcopy(validate_feature_manifest(manifest, data))
    columns = manifest["selected_indices"]
    config = asdict(model_config) if hasattr(model_config, "__dataclass_fields__") else dict(model_config)
    if (config.get("input_dim") != len(columns)
            or config.get("seq_len") != data["X"].shape[1]):
        raise ValueError("Model dimensions must match the selected features and full time axis.")
    if not isinstance(run_id, str) or not run_id:
        raise ValueError("A nonempty run_id is required.")
    settings = _experiment_settings(settings, model_kind)
    seed_everything(settings["seed"])
    counts = _experiment_class_counts(data)
    pos_weight = counts["train"]["negatives"] / counts["train"]["positives"]
    device = resolve_device(settings.get("device", "auto")) if model_kind == "transformer" else torch.device("cpu")
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device))
    loaders = {split: _experiment_loader(data, split, columns, settings["batch_size"], settings["seed"])
               for split in ("train", "validation")}
    history, best_ap, best_epoch, n_iterations = [], -np.inf, None, None
    if model_kind == "transformer":
        model = ClaimsTransformer(ModelConfig(**config)).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"])
        online_loader = _experiment_loader(data, "train", columns, settings["batch_size"], settings["seed"], True)
        patience_reference, without_progress, best_state = -np.inf, 0, None
        print(f"{run_id}: {len(columns)} features; device={device}; TRAIN positive weight={pos_weight:.4f}", flush=True)
        for epoch in range(1, settings["epochs"] + 1):
            model.train()
            online_sum, n = 0.0, 0
            for raw, y in online_loader:
                raw, y = raw.to(device), y.to(device)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(_experiment_forward(model, raw, model_kind), y)
                if not torch.isfinite(loss):
                    raise ValueError("Nonfinite optimization loss.")
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), settings["grad_clip"], error_if_nonfinite=True)
                optimizer.step()
                online_sum += float(loss.item()) * len(y)
                n += len(y)
            train_loss, train_y, train_scores = _experiment_predict_loader(model, loaders["train"], device, model_kind, criterion)
            val_loss, val_y, val_scores = _experiment_predict_loader(model, loaders["validation"], device, model_kind, criterion)
            val_ap = float(average_precision_score(val_y, val_scores))
            history.append({"epoch": epoch, "optimization_loss": online_sum / n,
                            "training_loss": train_loss, "validation_loss": val_loss,
                            "training_average_precision": float(average_precision_score(train_y, train_scores)),
                            "training_roc_auc": float(roc_auc_score(train_y, train_scores)),
                            "validation_average_precision": val_ap,
                            "validation_roc_auc": float(roc_auc_score(val_y, val_scores))})
            if val_ap > best_ap:
                best_ap, best_epoch = val_ap, epoch
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            if val_ap > patience_reference + settings["min_delta"]:
                patience_reference, without_progress = val_ap, 0
            else:
                without_progress += 1
            print(f"Epoch {epoch:02d}: eval TRAIN loss={train_loss:.5f}; VAL loss={val_loss:.5f}; VAL AP={val_ap:.5f}", flush=True)
            if without_progress >= settings["patience"]:
                print(f"Early stopping; restoring epoch {best_epoch}.", flush=True)
                break
        model.load_state_dict(best_state, strict=True)
        parameter_count = sum(parameter.numel() for parameter in model.parameters())
        transform = "selected_monthly_log1p"
        preprocessing = "Selected raw monthly counts -> log1p once per batch; no fitted scaler."
    else:
        aggregate = _logistic_train_matrix(data, columns, settings["batch_size"])
        scaler = StandardScaler().fit(aggregate)
        standardized = scaler.transform(aggregate)
        estimator = LogisticRegression(C=settings["logistic_C"], class_weight="balanced", solver="lbfgs",
                                       max_iter=settings["logistic_max_iter"], random_state=settings["seed"])
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            estimator.fit(standardized, data["y"][data["indices"]["train"]])
        for warning in caught:
            if issubclass(warning.category, ConvergenceWarning):
                raise RuntimeError("Logistic regression did not converge; no checkpoint was accepted. Increase logistic_max_iter or review inputs.")
            warnings.warn(str(warning.message), warning.category, stacklevel=2)
        if list(estimator.classes_) != [0, 1]:
            raise ValueError("Unexpected logistic class order.")
        n_iterations = int(np.max(estimator.n_iter_))
        model = NumericLogisticModel(estimator.coef_[0], estimator.intercept_, scaler.mean_, scaler.scale_, config["seq_len"])
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        parameter_count = len(columns) + 1
        transform = "selected_monthly_sum_log1p_train_standardize"
        preprocessing = "Selected monthly raw counts -> sum across months -> log1p -> StandardScaler fitted on TRAIN only; L2 logistic regression, balanced class weights. Month order is discarded."
        del aggregate, standardized, estimator, scaler
    # These final diagnostics describe the restored best model, not the last epoch.
    train_loss, train_y, train_scores = _experiment_predict_loader(model, loaders["train"], device, model_kind, criterion)
    val_loss, val_y, val_scores = _experiment_predict_loader(model, loaders["validation"], device, model_kind, criterion)
    threshold = float(select_validation_threshold(val_y, val_scores))
    best_ap = float(average_precision_score(val_y, val_scores))
    if model_kind == "logistic":
        history.append({"epoch": None, "optimization_loss": None,
                        "training_loss": train_loss, "validation_loss": val_loss,
                        "training_average_precision": float(average_precision_score(train_y, train_scores)),
                        "training_roc_auc": float(roc_auc_score(train_y, train_scores)),
                        "validation_average_precision": best_ap,
                        "validation_roc_auc": float(roc_auc_score(val_y, val_scores))})
    summary = {"run_id": run_id, "training_complete": True, "model_kind": model_kind,
               "completed_at_utc": datetime.now(timezone.utc).isoformat(),
               "best_epoch": best_epoch, "epochs_completed": len(history) if model_kind == "transformer" else 0,
               "optimizer_iterations": n_iterations,
               "best_validation_average_precision": best_ap, "validation_threshold": threshold,
               "training_metrics": {**classification_metrics(train_y, train_scores, threshold), "loss": train_loss},
               "validation_metrics": {**classification_metrics(val_y, val_scores, threshold), "loss": val_loss},
               "training_topk": _experiment_topk(data, "train", train_y, train_scores),
               "validation_topk": _experiment_topk(data, "validation", val_y, val_scores),
               "train_pos_weight": float(pos_weight), "class_counts": counts,
               "model_parameter_count": parameter_count, "model_config": config,
               "training_settings": settings, "feature_manifest": manifest,
               "feature_manifest_sha256": manifest["manifest_sha256"],
               "selected_feature_count": len(columns), "input_hashes": copy.deepcopy(data["hashes"]),
               "resolved_device": str(device), "preprocessing": preprocessing,
               "loss_definition": "Snapshot-average binary cross-entropy weighted by TRAIN negatives/positives; TRAIN and VALIDATION diagnostic losses both use eval mode.",
               "test_inference_performed": False,
               "source_review": "Structural checks only; clinical cutoff, outcome-event exclusion and historical claims availability are not certified by these notebooks.",
               "runtime": runtime_metadata()}
    payload = {"format_version": 2, "training_complete": True, "run_id": run_id,
               "model_kind": model_kind, "model_state_dict": best_state, "model_config": config,
               "training_settings": settings, "input_hashes": copy.deepcopy(data["hashes"]),
               "feature_names": list(data["features"]), "feature_manifest": manifest,
               "time_steps": list(range(config["seq_len"])), "tensor_shape": [int(value) for value in data["X"].shape],
               "transform": transform, "validation_threshold": threshold,
               "selection_metric": "validation_average_precision", "best_epoch": best_epoch,
               "best_validation_average_precision": best_ap}
    buffer = io.BytesIO()
    torch.save(payload, buffer)
    return buffer.getvalue(), summary, pd.DataFrame(history)


def _verify_experiment_payload(payload, data, run_id):
    if (not isinstance(payload, dict) or payload.get("format_version") != 2
            or payload.get("training_complete") is not True or payload.get("run_id") != run_id):
        raise ValueError("Checkpoint is incomplete, unsupported, or belongs to another run.")
    if payload.get("input_hashes") != data["hashes"]:
        raise ValueError("Full tensor content, keys, labels, feature order or frozen split changed after training.")
    if (payload.get("feature_names") != list(data["features"])
            or payload.get("time_steps") != list(range(data["X"].shape[1]))
            or payload.get("tensor_shape") != list(data["X"].shape)):
        raise ValueError("Checkpoint full tensor layout differs from current inputs.")
    manifest = validate_feature_manifest(payload.get("feature_manifest"), data)
    config = payload.get("model_config")
    if not isinstance(config, dict) or config.get("input_dim") != len(manifest["selected_indices"]) or config.get("seq_len") != data["X"].shape[1]:
        raise ValueError("Saved model dimensions disagree with the selected feature manifest.")
    threshold = payload.get("validation_threshold")
    if not isinstance(threshold, (int, float)) or not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("Missing valid frozen validation threshold.")
    expected_transform = {"transformer": "selected_monthly_log1p",
                          "logistic": "selected_monthly_sum_log1p_train_standardize"}
    if payload.get("model_kind") not in expected_transform or payload.get("transform") != expected_transform[payload["model_kind"]]:
        raise ValueError("Unknown or inconsistent preprocessing/model kind.")
    return manifest


def restore_experiment(blob, data, run_id, device="auto"):
    payload = torch.load(io.BytesIO(blob), map_location="cpu", weights_only=True)
    manifest = _verify_experiment_payload(payload, data, run_id)
    kind, state = payload["model_kind"], payload["model_state_dict"]
    if kind == "transformer":
        model = ClaimsTransformer(ModelConfig(**payload["model_config"]))
    else:
        expected = {"coefficient", "intercept", "scaler_mean", "scaler_scale"}
        if not isinstance(state, dict) or set(state) != expected or any(not torch.is_tensor(value) for value in state.values()):
            raise ValueError("Logistic checkpoint must contain only the expected numeric tensors.")
        model = NumericLogisticModel(state["coefficient"], state["intercept"], state["scaler_mean"], state["scaler_scale"], payload["model_config"]["seq_len"])
        if model.input_dim != len(manifest["selected_indices"]):
            raise ValueError("Logistic coefficients disagree with the frozen feature selection.")
    model.load_state_dict(state, strict=True)
    if any(not torch.isfinite(value).all() for value in model.state_dict().values()):
        raise ValueError("Checkpoint contains nonfinite model weights.")
    resolved = resolve_device(device)
    model.to(resolved).eval()
    return model, payload, resolved


def predict_experiment(model, payload, data, split, device):
    """Explicit inference entry point; TEST is allowed here, never in training."""
    manifest = _verify_experiment_payload(payload, data, payload.get("run_id"))
    if split not in ("train", "validation", "test"):
        raise ValueError("Unknown inference split.")
    loader = _experiment_loader(data, split, manifest["selected_indices"],
                                payload["training_settings"]["batch_size"], payload["training_settings"]["seed"])
    _, labels, scores = _experiment_predict_loader(model, loader, device, payload["model_kind"])
    return labels, scores


"""Controlled experiment recipes and aggregate-only reporting, embedded in notebooks."""
import copy
import hashlib
import io
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


TRAINING_ARTIFACT_NAMES = {
    "checkpoint.pt", "training_summary.json", "training_history.csv",
    "training_history.png", "feature_manifest.json", "feature_selection.csv",
    "training_topk.csv", "validation_topk.csv",
}
SELECTION_ARTIFACT_NAMES = {
    "selection.json", "comparison.csv", "comparison.png", "experiment_plan.json",
}
DEFAULT_EXPERIMENTS = [
    "all_baseline", "reduced150", "reduced150_dropout", "reduced150_decay",
    "reduced150_small", "reduced150_logistic",
]


def experiment_recipes(suite_id, names, seed=42):
    if not isinstance(suite_id, str) or not re.fullmatch(r"[A-Z][A-Z0-9_]{0,19}", suite_id):
        raise ValueError("SUITE_ID must be 1-20 uppercase letters, digits or underscores, starting with a letter.")
    if not isinstance(names, (list, tuple)) or not names or len(set(names)) != len(names):
        raise ValueError("Choose a nonempty list of distinct experiment names.")
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("Training seed must be an integer in [0, 2**32).")
    model = dict(seq_len=12, d_model=128, n_heads=4, encoder_layers=2,
                 feedforward_dim=256, dropout=0.2)
    training = dict(seed=seed, epochs=20, patience=5, min_delta=1e-4,
                    batch_size=64, learning_rate=1e-3, weight_decay=1e-4,
                    grad_clip=1.0, device="auto", logistic_C=1.0,
                    logistic_max_iter=2000)
    catalog = {}
    for base, mode in (("all_baseline", "all"), ("reduced150", "reduced"), ("core", "core")):
        catalog[base] = dict(feature_mode=mode, model_kind="transformer",
                             model_settings=copy.deepcopy(model), training_settings=copy.deepcopy(training),
                             reference="all_baseline" if base != "all_baseline" else None)
        if base == "all_baseline":
            continue
        changes = {
            "dropout": ("model_settings", {"dropout": 0.3}),
            "decay": ("training_settings", {"weight_decay": 1e-3}),
            "small": ("model_settings", {"d_model": 64, "encoder_layers": 1, "feedforward_dim": 128}),
            "low_lr": ("training_settings", {"learning_rate": 3e-4}),
            "logistic": (None, {}),
        }
        for suffix, (section, delta) in changes.items():
            item = copy.deepcopy(catalog[base])
            item["reference"] = base
            if section:
                item[section].update(delta)
            else:
                item["model_kind"] = "logistic"
            catalog[f"{base}_{suffix}"] = item
    unknown = sorted(set(names) - set(catalog))
    if unknown:
        raise ValueError(f"Unknown experiments {unknown}; available: {sorted(catalog)}")
    recipes = []
    for name in names:
        recipe = copy.deepcopy(catalog[name])
        recipe.update(name=name, run_id=f"{suite_id}_{name.upper()}")
        recipes.append(recipe)
    return recipes


def experiment_signature(recipe, manifest, data, implementation_sha256):
    return digest_json({"recipe": recipe, "feature_manifest": manifest,
                        "input_hashes": data["hashes"], "implementation_sha256": implementation_sha256})


def experiment_history_figure(history, summary):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    if summary["model_kind"] == "logistic":
        for ax in axes:
            ax.axis("off")
        axes[0].text(0.05, .65, "Regularized logistic regression\nTRAIN-fitted scaling; 12-month count totals\nNo Transformer epochs / dropout", fontsize=12)
        axes[1].text(.05, .65, f"Validation AP: {summary['validation_metrics']['average_precision']:.4f}\n"
                      f"Training AP: {summary['training_metrics']['average_precision']:.4f}", fontsize=12)
    else:
        axes[0].plot(history.epoch, history.training_loss, label="TRAIN, evaluation mode")
        axes[0].plot(history.epoch, history.validation_loss, label="VALIDATION, evaluation mode")
        axes[0].set(xlabel="Epoch", ylabel="TRAIN-weighted BCE", title="Comparable loss (dropout disabled)")
        axes[0].legend(fontsize=8)
        axes[1].plot(history.epoch, history.training_average_precision, label="TRAIN")
        axes[1].plot(history.epoch, history.validation_average_precision, marker="o", label="VALIDATION")
        axes[1].axvline(summary["best_epoch"], color="gray", linestyle="--")
        axes[1].set(xlabel="Epoch", ylabel="Average precision", title="Select checkpoint on validation AP")
        axes[1].legend(fontsize=8)
        for ax in axes:
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.suptitle(summary["run_id"])
    return fig


def training_artifacts_for_run(blob, summary, history, manifest, feature_audit):
    fig = experiment_history_figure(history, summary)
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=140)
    plt.close(fig)
    return {
        "checkpoint.pt": blob,
        "training_summary.json": canonical_json(summary).encode("utf-8"),
        "training_history.csv": history.to_csv(index=False).encode("utf-8"),
        "training_history.png": buffer.getvalue(),
        "feature_manifest.json": canonical_json(manifest).encode("utf-8"),
        "feature_selection.csv": feature_audit.to_csv(index=False).encode("utf-8"),
        "training_topk.csv": pd.DataFrame(summary["training_topk"]).to_csv(index=False).encode("utf-8"),
        "validation_topk.csv": pd.DataFrame(summary["validation_topk"]).to_csv(index=False).encode("utf-8"),
    }


def checked_saved_training(artifacts, data, recipe, manifest, signature):
    summary = json.loads(artifacts["training_summary.json"])
    saved_manifest = json.loads(artifacts["feature_manifest.json"])
    if saved_manifest != manifest or summary.get("experiment_signature") != signature:
        raise ValueError("Existing run has different settings/features/code/inputs. Choose a new SUITE_ID.")
    if summary.get("test_inference_performed") is not False or summary.get("training_complete") is not True:
        raise ValueError("Training record does not certify TEST exclusion.")
    model, payload, _ = restore_experiment(artifacts["checkpoint.pt"], data, recipe["run_id"], "cpu")
    del model
    if (summary.get("validation_threshold") != payload.get("validation_threshold")
            or payload.get("feature_manifest") != manifest
            or summary.get("input_hashes") != data["hashes"]
            or summary.get("model_kind") != recipe["model_kind"]
            or summary.get("run_id") != recipe["run_id"]
            or payload.get("model_kind") != recipe["model_kind"]
            or payload.get("model_config") != recipe["model_settings"]
            or payload.get("training_settings") != recipe["training_settings"]
            or summary.get("model_config") != payload.get("model_config")
            or summary.get("training_settings") != payload.get("training_settings")
            or summary.get("feature_manifest") != manifest
            or summary.get("feature_manifest_sha256") != manifest["manifest_sha256"]
            or summary.get("best_validation_average_precision") != payload.get("best_validation_average_precision")
            or summary.get("validation_metrics", {}).get("average_precision") != payload.get("best_validation_average_precision")
            or summary.get("best_epoch") != payload.get("best_epoch")):
        raise ValueError("Saved training summary and checkpoint disagree.")
    return summary


def comparison_frame(run_summaries):
    rows = []
    for summary in run_summaries:
        val = summary["validation_metrics"]
        row = {
            "run_id": summary["run_id"], "model_kind": summary["model_kind"],
            "feature_mode": summary["feature_manifest"]["mode"],
            "n_features": len(summary["feature_manifest"]["selected_indices"]),
            "parameter_count": summary["model_parameter_count"],
            "best_epoch": summary["best_epoch"],
            "training_ap": summary["training_metrics"]["average_precision"],
            "validation_ap": val["average_precision"], "validation_roc_auc": val["roc_auc"],
            "validation_precision": val["precision"], "validation_recall": val["recall"],
            "validation_f1": val["f1"], "validation_threshold": summary["validation_threshold"],
        }
        for top in summary["validation_topk"]:
            pct = int(top["top_k_pct"])
            row[f"validation_lift_{pct}"] = top["lift"]
            row[f"validation_recall_{pct}"] = top["recall"]
        rows.append(row)
    result = pd.DataFrame(rows)
    if result.empty or result.run_id.duplicated().any() or not np.isfinite(result.validation_ap).all():
        raise ValueError("Need unique, completed runs with finite validation AP.")
    return result.sort_values(["validation_ap", "run_id"], ascending=[False, True]).reset_index(drop=True)


def comparison_figure(frame):
    fig, axes = plt.subplots(1, 2, figsize=(13, max(4, .55 * len(frame))), constrained_layout=True)
    labels = frame.run_id.to_list()
    y = np.arange(len(frame))
    axes[0].barh(y - .17, frame.training_ap, height=.32, label="TRAIN")
    axes[0].barh(y + .17, frame.validation_ap, height=.32, label="VALIDATION")
    axes[0].set(yticks=y, yticklabels=labels, xlabel="Average precision", title="Validation selects the run")
    axes[0].invert_yaxis()
    axes[0].legend(loc="upper center", bbox_to_anchor=(.5, -.20), ncol=2)
    axes[1].barh(y, frame.validation_lift_10, height=.55, color="#007E87")
    axes[1].set(yticks=y, yticklabels=labels, xlabel="Top-10% lift", title="VALIDATION targeting performance")
    axes[1].invert_yaxis()
    axes[1].axvline(1, color="gray", linestyle="--")
    return fig


def verify_preparation_audits(data, preparation, split_audit):
    if not isinstance(preparation, dict) or not isinstance(split_audit, dict):
        raise ValueError("Preparation and split audits must be completed JSON records.")
    if (preparation.get("mode") != "reuse_and_validate_saved_v63_tensor"
            or split_audit.get("mode") != "reuse_original_patient_split"
            or split_audit.get("new_assignments_created") is not False):
        raise ValueError("Expected the preparation and original split audits from notebooks 01 and 02.")
    if preparation.get("tensor_shape") != list(data["X"].shape):
        raise ValueError("Tensor shape differs from the saved preparation audit.")
    for key in ("model_input_sha256", "feature_names_sha256"):
        if preparation.get(key) != data["hashes"][key]:
            raise ValueError(f"Preparation audit differs on {key}.")
    for key in ("snapshot_manifest_sha256", "split_config_sha256", "feature_names_sha256"):
        if split_audit.get(key) != data["hashes"][key]:
            raise ValueError(f"Frozen split audit differs on {key}.")
    source_rows = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP)]
                   for r in data["metadata"].itertuples()]
    source_hash = digest_json(source_rows)
    if any(report.get("source_snapshots_sha256") != source_hash for report in (preparation, split_audit)):
        raise ValueError("Preparation/split audit snapshot identity differs.")
    if preparation.get("source_prefix") != split_audit.get("source_prefix"):
        raise ValueError("Preparation and split audit use different sources.")


def release_experiment_inputs(data):
    if data is not None and hasattr(data.get("X"), "_mmap") and not data["X"]._mmap.closed:
        data["X"]._mmap.close()
        data["temporary_directory"].cleanup()

IMPLEMENTATION_SHA256 = "ec80a5932cc98cdab46ec9d0f195d3df8fd44f7178ce61126207cca9dc548011"

"""DL-POC aggregate evaluation, suitable for an inline notebook cell.

No repository imports, file writes, threshold tuning, or patient-level exports.
Rates are fractions; lift is relative to the overall evaluated response rate.
"""

import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, precision_recall_curve, roc_curve,
)


def _require(condition, message):
    if not condition:
        raise ValueError(message)


def _real_array(values, name):
    raw = np.asarray(values)
    _require(not np.iscomplexobj(raw) and not (
        raw.dtype == object
        and any(isinstance(value, (complex, np.complexfloating)) for value in raw.flat)
    ), f"{name} must be real, not complex.")
    try:
        return np.asarray(raw, dtype=float)
    except (TypeError, ValueError, OverflowError):
        raise ValueError(f"{name} must contain numeric values.") from None


def _checked_labels_scores(y, scores):
    labels = _real_array(y, "Labels")
    probabilities = _real_array(scores, "Probabilities")
    _require(labels.ndim == probabilities.ndim == 1
             and len(labels) == len(probabilities) and len(labels) > 0,
             "Labels and probabilities must be aligned, nonempty 1-D arrays.")
    _require(np.isfinite(labels).all() and np.isin(np.asarray(y), [0, 1]).all(),
             "Labels must contain only finite binary 0/1 values.")
    _require(np.isfinite(probabilities).all()
             and ((probabilities >= 0) & (probabilities <= 1)).all(),
             "Probabilities must be finite and within [0, 1].")
    return labels.astype(np.int64), probabilities


def classification_metrics(y, scores, threshold):
    """Use a threshold already selected on VALIDATION; never tune on TEST."""
    labels, probabilities = _checked_labels_scores(y, scores)
    cutoff = _real_array(threshold, "Threshold")
    _require(cutoff.ndim == 0 and np.isfinite(cutoff).all()
             and 0 <= float(cutoff) <= 1,
             "Threshold must be a finite scalar in [0, 1].")
    cutoff = float(cutoff)
    predicted = (probabilities >= cutoff).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(labels, probabilities))
        if labels.sum() else None,
        "roc_auc": float(roc_auc_score(labels, probabilities))
        if len(np.unique(labels)) == 2 else None,
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "threshold": cutoff,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def _checked_scored_snapshots(frame, score_column):
    _require(isinstance(frame, pd.DataFrame), "Evaluation input must be a pandas DataFrame.")
    _require(isinstance(score_column, str), "Score column must be a column name.")
    _require(frame.columns.is_unique, "Evaluation input contains duplicate column names.")
    keys = ["PATIENT_ID", "END_DT"]
    _require(score_column not in keys + ["RESP"], "Score column must differ from metadata columns.")
    _require(set(keys + ["RESP", score_column]).issubset(frame.columns),
             "Evaluation input is missing snapshot keys, RESP, or scores.")
    out = frame[keys + ["RESP", score_column]].copy()
    _require(not out[keys].isna().any().any(), "Snapshot keys must not be missing.")
    _require(out.PATIENT_ID.map(lambda value: isinstance(value, str)).all(),
             "PATIENT_ID must contain strings to preserve literal identifiers.")
    out["PATIENT_ID"] = out.PATIENT_ID.astype(str)
    _require(out.PATIENT_ID.str.len().gt(0).all()
             and out.PATIENT_ID.str.strip().eq(out.PATIENT_ID).all(),
             "Patient keys must be nonempty with no surrounding whitespace.")
    dates = out.END_DT.astype(str)
    _require(dates.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all(),
             "END_DT must contain ISO calendar dates (YYYY-MM-DD), not intraday timestamps.")
    parsed = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
    _require(parsed.notna().all(), "END_DT contains invalid calendar dates.")
    out["END_DT"] = parsed.dt.strftime("%Y-%m-%d")
    _require(not out.duplicated(keys).any(), "Duplicate patient/date snapshots are not allowed.")
    labels, scores = _checked_labels_scores(out.RESP.to_numpy(), out[score_column].to_numpy())
    out["RESP"] = labels
    out[score_column] = scores
    _require(len(out) >= 10, "At least 10 snapshots are required for ten nonempty deciles.")
    return out


def _snapshot_hash(patient, end_date):
    # This public salt gives deterministic, label-independent ties, not anonymity.
    payload = json.dumps(["tak861-targeting-v1", patient, end_date], separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _score_order(frame, score_column):
    # Labels are deliberately inaccessible to ordering logic.
    scoring = frame[["PATIENT_ID", "END_DT", score_column]].rename(
        columns={score_column: "_probability"}
    ).copy()
    scoring["_tie"] = [
        _snapshot_hash(patient, date)
        for patient, date in scoring[["PATIENT_ID", "END_DT"]].itertuples(index=False, name=None)
    ]
    scoring["_position"] = np.arange(len(scoring))
    return scoring.sort_values(
        ["_probability", "_tie", "PATIENT_ID", "END_DT"],
        ascending=[False, True, True, True],
    )["_position"].to_numpy()


def _rank_snapshots(checked, score_column):
    ranked = checked.iloc[_score_order(checked, score_column)].reset_index(drop=True)
    n = len(ranked)
    ranks = np.arange(1, n + 1, dtype=np.int64)
    boundaries = (np.arange(1, 11, dtype=np.int64) * n + 9) // 10
    ranked["decile"] = np.searchsorted(boundaries, ranks, side="left") + 1
    return ranked


def _decile_table(ranked, model_name):
    n, positives = len(ranked), int(ranked.RESP.sum())
    overall_rate = positives / n
    table = ranked.groupby("decile").agg(
        n_snapshots=("RESP", "size"), n_resp1=("RESP", "sum")
    ).reindex(range(1, 11)).reset_index()
    table.insert(0, "model", model_name)
    table["population_fraction"] = table.n_snapshots / n
    table["response_rate"] = table.n_resp1 / table.n_snapshots
    table["decile_precision"] = table.response_rate
    table["share_of_all_resp1"] = table.n_resp1 / positives if positives else np.nan
    table["cumulative_n_snapshots"] = table.n_snapshots.cumsum()
    table["cumulative_population_fraction"] = table.cumulative_n_snapshots / n
    table["cumulative_resp1"] = table.n_resp1.cumsum()
    table["cumulative_recall"] = table.cumulative_resp1 / positives if positives else np.nan
    table["cumulative_precision"] = table.cumulative_resp1 / table.cumulative_n_snapshots
    table["decile_lift"] = table.response_rate / overall_rate if positives else np.nan
    table["cumulative_lift"] = table.cumulative_precision / overall_rate if positives else np.nan
    table["overall_test_response_rate"] = overall_rate
    return table


def _topk_table(deciles):
    selected = deciles.loc[deciles.decile.isin([1, 2, 3])].copy()
    selected["top_k_pct"] = selected.decile * 10
    return selected.rename(columns={
        "cumulative_n_snapshots": "n_selected", "cumulative_resp1": "n_resp1_selected",
        "cumulative_recall": "recall", "cumulative_precision": "precision",
        "cumulative_lift": "lift", "cumulative_population_fraction": "actual_population_fraction",
    })[["model", "top_k_pct", "n_selected", "n_resp1_selected", "recall", "precision",
        "lift", "actual_population_fraction"]].rename(
        columns={"n_resp1_selected": "n_resp1"}
    ).reset_index(drop=True)


def _tie_table(ranked, score_column, model_name):
    scores = ranked[score_column].to_numpy()
    rows = []
    for percentage in (10, 20, 30):
        cutoff = (percentage * len(scores) + 99) // 100
        value = scores[cutoff - 1]
        total = int((scores == value).sum())
        included = int((scores[:cutoff] == value).sum())
        rows.append({
            "model": model_name, "top_k_pct": percentage, "cutoff_score": value,
            "n_tied_at_cutoff": total, "n_tied_selected": included,
            "tie_crosses_boundary": included < total,
        })
    return pd.DataFrame(rows)


def _evaluation_figures(results, checked, score_column):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter

    table, topk = results["deciles"], results["topk"]
    model_name = results["global_metrics"].iloc[0]["model"]
    has_positives = bool(results["global_metrics"].iloc[0]["n_resp1"])
    figures = {}
    color = "#007E87"
    with plt.rc_context({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                         "figure.facecolor": "white", "axes.facecolor": "white"}):
        fig, ax = plt.subplots(figsize=(8, 5.5), layout="constrained")
        if has_positives:
            ax.plot(np.r_[0, table.cumulative_population_fraction],
                    np.r_[0, table.cumulative_recall], marker="o", markersize=4,
                    linewidth=2.3, color=color, label=model_name)
            ax.plot([0, 1], [0, 1], linestyle="--", color="#777777", label="Random selection")
            ax.legend(loc="lower right")
        else:
            ax.text(.5, .5, "Recall undefined: no positive TEST snapshots", ha="center", transform=ax.transAxes)
        ax.set(xlim=(0, 1), ylim=(0, 1.02), xlabel="TEST snapshots selected (actual fraction)",
               ylabel="Cumulative RESP=1 captured / recall", title="Cumulative gains | held-out TEST snapshots")
        ax.xaxis.set_major_formatter(PercentFormatter(1))
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.grid(alpha=.18)
        figures["gains"] = fig

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), layout="constrained")
        if has_positives:
            axes[0].bar(table.decile, table.decile_lift, width=.65, color=color)
            axes[1].plot(table.cumulative_population_fraction, table.cumulative_lift,
                         color=color, marker="o", markersize=4, linewidth=2)
        for ax in axes:
            if has_positives:
                ax.axhline(1, color="#777777", linestyle="--", linewidth=1, label="Population baseline = 1")
            else:
                ax.text(.5, .5, "Lift undefined: no positive snapshots", ha="center", transform=ax.transAxes)
            ax.set_ylabel("Lift vs overall TEST response rate")
            ax.grid(axis="y", alpha=.18)
            ax.set_ylim(bottom=0)
        axes[0].set(xticks=range(1, 11), xlabel="Decile (1 = highest propensity)", title="Within-decile lift")
        axes[1].set(xlabel="TEST snapshots selected (actual fraction)", title="Cumulative lift")
        axes[1].xaxis.set_major_formatter(PercentFormatter(1))
        if has_positives:
            axes[1].legend(fontsize=9)
        figures["lift"] = fig

        fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), layout="constrained")
        for ax, metric, title in zip(axes, ["recall", "precision", "lift"],
                                     ["Positive snapshots captured", "Precision in selected snapshots", "Cumulative lift"]):
            values = topk[metric].to_numpy(dtype=float)
            bars = ax.bar(np.arange(3), np.nan_to_num(values, nan=0), width=.60, color=color)
            labels = [(f"{value:.2f}×" if metric == "lift" else f"{value:.1%}")
                      if np.isfinite(value) else "N/A" for value in values]
            ax.bar_label(bars, labels=labels, padding=3, fontsize=8)
            ax.set(xticks=range(3), xticklabels=["Top 10%", "Top 20%", "Top 30%"], title=title)
            ax.set_ylim(bottom=0, top=max(ax.get_ylim()[1] * 1.16, .01))
            if metric != "lift":
                ax.yaxis.set_major_formatter(PercentFormatter(1))
                if ax.get_ylim()[1] > 1:
                    ax.set_yticks(np.arange(0, 1.001, .2))
            ax.grid(axis="y", alpha=.18)
        fig.suptitle(f"{model_name} | held-out TEST targeting performance", fontsize=13)
        figures["topk_performance"] = fig

        fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
        bars = ax.bar(table.decile, table.response_rate, color=color)
        ax.bar_label(bars, labels=[f"{value:.1%}" for value in table.response_rate], padding=3, fontsize=9)
        ax.axhline(float(checked.RESP.mean()), color="#777777", linestyle="--",
                   label="Overall TEST response rate")
        ax.set(xticks=range(1, 11), xlabel="Decile (1 = highest propensity)",
               ylabel="Observed response rate", title=f"{model_name} | response rate by decile",
               ylim=(0, max(table.response_rate.max() * 1.22, .01)))
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        if ax.get_ylim()[1] > 1:
            ax.set_yticks(np.arange(0, 1.001, .2))
        ax.legend()
        figures["response_rate"] = fig

        metrics = results["global_metrics"].iloc[0]
        y, scores = checked.RESP.to_numpy(), checked[score_column].to_numpy()
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.9), layout="constrained")
        if has_positives:
            precision, recall, _ = precision_recall_curve(y, scores)
            axes[0].step(recall, precision, where="post", color=color,
                         label=f"Average precision = {metrics['average_precision']:.3f}")
            axes[0].axhline(y.mean(), linestyle="--", color="#777777", label="TEST prevalence")
            axes[0].legend()
        else:
            axes[0].text(.5, .5, "Undefined: no positive TEST snapshots", ha="center", transform=axes[0].transAxes)
        if len(np.unique(y)) == 2:
            fpr, tpr, _ = roc_curve(y, scores)
            axes[1].plot(fpr, tpr, color=color, label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
            axes[1].plot([0, 1], [0, 1], linestyle="--", color="#777777")
            axes[1].legend(loc="lower right")
        else:
            axes[1].text(.5, .5, "Undefined: only one TEST class", ha="center", transform=axes[1].transAxes)
        axes[0].set(xlabel="Recall", ylabel="Precision", title="Precision–recall (primary)",
                    xlim=(0, 1), ylim=(0, 1.02))
        axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC (supporting)",
                    xlim=(0, 1), ylim=(0, 1.02))
        fig.suptitle("AP is the non-interpolated average precision, not trapezoidal PR-AUC.", fontsize=10)
        figures["discrimination"] = fig

        matrix = np.array([[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]], dtype=int)
        fig, ax = plt.subplots(figsize=(5.2, 4.6), layout="constrained")
        ax.imshow(matrix, cmap="Blues", vmin=0)
        for (row, column), value in np.ndenumerate(matrix):
            ax.text(column, row, f"{value:,}", ha="center", va="center", fontsize=18,
                    color="white" if value > matrix.max() / 2 else "#183B4E")
        ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["RESP=0", "RESP=1"],
               yticklabels=["RESP=0", "RESP=1"], xlabel="Predicted", ylabel="Actual",
               title=f"TEST confusion matrix\nVALIDATION threshold = {metrics['threshold']:.4f}")
        figures["confusion_matrix"] = fig
    return figures


def evaluate_scores(scored_snapshots, threshold, *, score_column="P_RESP1",
                    model_name="Transformer", make_plots=True):
    """Return aggregate frames and figures for an already verified held-out cohort.

    Input: pandas DataFrame with PATIENT_ID (literal string), END_DT (ISO date or
    Python date), RESP, and score_column. threshold must come from VALIDATION.
    Caller must verify exact TEST keys/labels and frozen model provenance before
    this call. Repeated snapshots contribute separately: metrics are snapshot-
    level, not unique-patient capture. Equal scores use a fixed key hash; decile
    boundaries are ceil(d*N/10). No patient rows are returned or persisted.

    Output keys: global_metrics, deciles, topk, ties (pandas DataFrames), and
    figures (gains, lift, topk_performance, response_rate, discrimination,
    confusion_matrix: matplotlib Figure; empty if disabled).
    Caller may display figures and then close them with plt.close(fig).
    """
    checked = _checked_scored_snapshots(scored_snapshots, score_column)
    metrics = classification_metrics(checked.RESP, checked[score_column], threshold)
    ranked = _rank_snapshots(checked, score_column)
    deciles = _decile_table(ranked, model_name)
    base = {
        "model": model_name, "n_test_snapshots": len(checked),
        "n_test_patients": int(checked.PATIENT_ID.nunique()),
        "n_resp1": int(checked.RESP.sum()), "test_response_rate": float(checked.RESP.mean()),
    }
    results = {
        "global_metrics": pd.DataFrame([{**base, **metrics}]),
        "deciles": deciles,
        "topk": _topk_table(deciles),
        "ties": _tie_table(ranked, score_column, model_name),
    }
    results["figures"] = _evaluation_figures(results, checked, score_column) if make_plots else {}
    return results


In [ ]:
# CELL 3 — Load the frozen selection and verify all original inputs
if not table_exists(SELECTION_TABLE):
    raise FileNotFoundError("Complete and save notebook 03 before evaluation.")
selection_artifacts = read_artifacts(SELECTION_TABLE, SELECTION_ARTIFACT_NAMES)
selection = json.loads(selection_artifacts["selection.json"])
if (selection.get("suite_id") != SUITE_ID or selection.get("selection_complete") is not True
        or selection.get("test_used_for_selection") is not False
        or selection.get("implementation_sha256") != IMPLEMENTATION_SHA256):
    raise ValueError("Missing, changed or incompatible frozen validation selection.")
RUN_ID = selection["selected_run_id"]
MODEL_TABLE = f"{EXPERIMENT_PREFIX}_MODEL_{RUN_ID}"
EVALUATION_TABLE = f"{EXPERIMENT_PREFIX}_EVAL_{RUN_ID}"
if selection.get("selected_model_table") != MODEL_TABLE:
    raise ValueError("Selected model table differs from this suite's namespace.")
if table_exists(EVALUATION_TABLE):
    raise FileExistsError("This selected run already has a saved TEST report. Reuse it instead of retuning on TEST.")
if not table_exists(MODEL_TABLE):
    raise FileNotFoundError("Selected completed model is missing.")
training_artifacts = read_artifacts(MODEL_TABLE, TRAINING_ARTIFACT_NAMES)
if hashlib.sha256(training_artifacts["checkpoint.pt"]).hexdigest() != selection["selected_checkpoint_sha256"]:
    raise ValueError("Selected checkpoint changed after validation selection.")
training_summary = json.loads(training_artifacts["training_summary.json"])

if "data" in globals():
    release_experiment_inputs(data)
for table in (PREPARATION_TABLE, SPLIT_AUDIT_TABLE):
    if not table_exists(table):
        raise FileNotFoundError("Run and save notebooks 01 and 02 before this stage.")
preparation = json.loads(read_artifacts(PREPARATION_TABLE, ["preparation_report.json"])["preparation_report.json"])
split_audit = json.loads(read_artifacts(SPLIT_AUDIT_TABLE, ["split_audit.json"])["split_audit.json"])
data = load_inputs()
try:
    verify_preparation_audits(data, preparation, split_audit)
except BaseException:
    release_experiment_inputs(data)
    raise
display(data["metadata"].groupby("SPLIT", observed=True).agg(
    patients=("PATIENT_ID", "nunique"), snapshots=("RESP", "size"), positive_snapshots=("RESP", "sum")))
print("Original tensor and patient assignments match the saved preparation audits.")

if selection.get("input_hashes") != data["hashes"]:
    release_experiment_inputs(data)
    raise ValueError("Inputs changed after validation selected the run.")


In [ ]:
# CELL 4 — Restore the saved feature selection, model and validation threshold
try:
    saved_manifest = json.loads(training_artifacts["feature_manifest.json"])
    saved_plan = json.loads(selection_artifacts["experiment_plan.json"])
    selected_recipes = [recipe for recipe in saved_plan if recipe["run_id"] == RUN_ID]
    if len(selected_recipes) != 1:
        raise ValueError("Selected run must occur exactly once in the saved experiment plan.")
    selected_recipe = selected_recipes[0]
    expected_signature = experiment_signature(selected_recipe, saved_manifest, data, IMPLEMENTATION_SHA256)
    training_summary = checked_saved_training(training_artifacts, data, selected_recipe, saved_manifest, expected_signature)
    model, checkpoint, device = restore_experiment(training_artifacts["checkpoint.pt"], data, RUN_ID, "auto")
    if (checkpoint["feature_manifest"]["manifest_sha256"] != selection["selected_feature_manifest_sha256"]
            or checkpoint["validation_threshold"] != selection["validation_threshold"]
            or checkpoint["best_validation_average_precision"] != selection["selected_validation_ap"]):
        raise ValueError("Frozen selection, summary and checkpoint disagree.")
except BaseException:
    release_experiment_inputs(data)
    raise
print("Model:", checkpoint["model_kind"], "| selected features:", len(saved_manifest["selected_indices"]))
print("Frozen validation threshold:", checkpoint["validation_threshold"])
print("No feature reselection, model fitting or threshold tuning occurs here.")


In [ ]:
# CELL 5 — Score the selected model on the original TEST cohort
test_y, test_scores = predict_experiment(model, checkpoint, data, "test", device)
test_rows = data["indices"]["test"]
scored_snapshots = data["metadata"].iloc[test_rows][["PATIENT_ID", "END_DT", "RESP"]].copy().reset_index(drop=True)
if not np.array_equal(test_y, scored_snapshots.RESP.to_numpy()):
    raise ValueError("Prediction labels lost alignment with snapshot keys.")
scored_snapshots["P_RESP1"] = test_scores
model_name = "DL-POC " + checkpoint["model_kind"] + " | " + RUN_ID
results = evaluate_scores(scored_snapshots, checkpoint["validation_threshold"], model_name=model_name)
evaluation_metadata = {
    "run_id": RUN_ID, "suite_id": SUITE_ID, "model_kind": checkpoint["model_kind"],
    "checkpoint_sha256": selection["selected_checkpoint_sha256"],
    "feature_manifest_sha256": saved_manifest["manifest_sha256"],
    "selected_features": saved_manifest["selected_features"],
    "input_hashes": data["hashes"], "selection_metric": selection["selection_metric"],
    "threshold_source": "frozen validation maximum-F1 threshold",
    "threshold": checkpoint["validation_threshold"], "evaluation_unit": "patient/date snapshot",
    "test_snapshots": len(scored_snapshots), "test_positive_snapshots": int(scored_snapshots.RESP.sum()),
    "ranking_ties": "label-independent snapshot-key hash", "test_used_for_selection": False,
    "test_status": selection["test_status"],
    "limitations": "Previously inspected original TEST; snapshot metrics; no confidence intervals; class-weighted scores are not calibrated probabilities.",
    "source_review": training_summary.get("source_review"),
    "business_target_specification": selection["business_target_specification"],
}
print("Selected-run TEST inference complete; original cohort is not a new confirmation holdout.")


In [ ]:
# CELL 6 — Review TEST targeting and classification reports
for name in ("global_metrics", "deciles", "topk", "ties"):
    print(name)
    display(results[name])
for name, figure in results["figures"].items():
    display(figure)
print("Metrics count snapshots, not unique patients. AP is non-interpolated average precision.")


In [ ]:
# CELL 7 — Save aggregate TEST results in the new namespace
evaluation_artifacts = {"evaluation_metadata.json": canonical_json(evaluation_metadata).encode("utf-8")}
for name in ("global_metrics", "deciles", "topk", "ties"):
    evaluation_artifacts[f"{name}.csv"] = results[name].to_csv(index=False).encode("utf-8")
for name, figure in results["figures"].items():
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", dpi=140, bbox_inches="tight")
    evaluation_artifacts[f"{name}.png"] = buffer.getvalue()
save_artifacts(EVALUATION_TABLE, evaluation_artifacts)
# No patient IDs or per-snapshot predictions are exported.


In [ ]:
# CELL 8 — Verify the report and release temporary inputs
if read_artifacts(EVALUATION_TABLE, evaluation_artifacts) != evaluation_artifacts:
    raise ValueError("Saved TEST report failed verification.")
release_experiment_inputs(data)
print("Saved and verified:", EVALUATION_TABLE)
print("Choose further changes on TRAIN/VALIDATION; use a fresh later-period cohort for final confirmation.")
